# P03 · A cohort prediction workflow with an unseen site

<!-- paper-first -->
### Research question

**Reading:** [PD02](../../curriculum/papers/design.md#pd02), [PM03](../../curriculum/papers/modeling.md#pm03). Review the assigned figure or result before starting the lesson.

**Question:** What evidence would show that a cohort predictor transfers to the held-out site rather than exploiting a shortcut?

Record a prediction, a source location, and one point you want this lesson to clarify. Ask your AI tutor to distinguish the paper’s evidence from its interpretation.
<!-- /paper-first -->

**Learning format:** predict → ask AI for one short operation → run → inspect → deliberately break → explain. Use Goose with a local Ollama model, or ChatGPT as a tutor and snippet writer. You are responsible for deciding whether the transformation answers the research question. Read the answer guide only after making your own prediction.

**Time:** 4–6 hours. **Prerequisites:** DS missingness/PCA/selection, modeling grouped validation and regularization. This experiment is synthetic so its ground truth is known; it is not evidence that imaging predicts a real clinical outcome.

A cohort analysis has several boundaries: rows from the same person, participants seen during model development, and institutions with different acquisition or recruitment patterns. A row-wise split can leak participant identity. A participant-wise split is better for new people at familiar sites, but it does not guarantee performance at a new site. The unit of generalization must match the scientific claim.

We construct repeated visits from three sites. Within the two development sites, an imaging proxy is correlated with the outcome through site membership. At the held-out site that shortcut fails. A second feature contains a modest stable signal. The strongest apparent relationship during development may therefore be a poor transportable predictor. This is a controlled demonstration of dataset shift, not a proposed scanner harmonization algorithm.

The workflow keeps the third site untouched, tunes ridge regularization using participant-grouped folds within development, and fits scaling only inside the pipeline. The final comparison includes a mean baseline and a simpler feature set. Any choice made after looking at the held-out scores is exploratory; a new external sample is needed to confirm the revised selection. The test set is not a reusable debugging set.

Use the [Neuromatch GLM model-selection sequence](https://compneuro.neuromatch.io/tutorials/W1D2_ModelFitting/student/W1D2_Tutorial6.html) and the [scikit-learn grouped cross-validation guide](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data) as deeper source work. Ask AI: **“Draw the outer held-out-site boundary and inner participant-fold boundary. Name each fitted step. Show a baseline, uncertainty at the participant level, and the claim supported by this exact split.”**


### Generate people, visits, and a site shortcut
Predict which feature will be stable at the new site.

In [1]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score
rng = np.random.default_rng(503)
n_people, visits = 180, 2
person = np.repeat(np.arange(n_people),visits)
site = np.repeat(np.repeat([0,1,2],60),visits)
latent = rng.normal(size=n_people)
y_person = latent + np.repeat([-2.,2.,0.],60) + rng.normal(0,0.3,n_people)
y = np.repeat(y_person,visits) + rng.normal(0,0.1,len(person))
stable = np.repeat(latent,visits) + rng.normal(0,0.3,len(person))
shortcut = np.choose(site,[-2.,2.,6.]) + rng.normal(0,0.15,len(person))
X = np.column_stack([stable,shortcut])
train,test = site < 2, site == 2
assert set(person[train]).isdisjoint(person[test])
print('Development people:',len(set(person[train])),'held-out people:',len(set(person[test])))

Development people: 120 held-out people: 60


### Tune only inside development
The same participant never appears on both sides of an inner fold.

In [2]:
cv = GroupKFold(n_splits=5)
for a,b in cv.split(X[train],y[train],person[train]):
    assert set(person[train][a]).isdisjoint(person[train][b])
search = GridSearchCV(make_pipeline(StandardScaler(),Ridge()),{'ridge__alpha':[0.1,1,10,100]},cv=cv,scoring='neg_mean_absolute_error')
search.fit(X[train],y[train],groups=person[train])
baseline = DummyRegressor(strategy='mean').fit(X[train],y[train])
stable_model = make_pipeline(StandardScaler(),Ridge(alpha=10)).fit(X[train,:1],y[train])
predictions = {'site-shortcut model':search.predict(X[test]),'mean baseline':baseline.predict(X[test]),'prespecified stable feature':stable_model.predict(X[test,:1])}
for name,pred in predictions.items():
    print(name,'MAE=',round(mean_absolute_error(y[test],pred),3),'R2=',round(r2_score(y[test],pred),3))
assert mean_absolute_error(y[test],predictions['site-shortcut model']) > mean_absolute_error(y[test],predictions['mean baseline'])

site-shortcut model MAE= 5.892 R2= -31.316
mean baseline MAE= 0.783 R2= -0.023
prespecified stable feature MAE= 0.344 R2= 0.833


### Bootstrap people, not visits
This interval describes uncertainty in this held-out site, not the distribution of all future sites.

In [3]:
errors = np.abs(y[test]-predictions['prespecified stable feature']).reshape(60,visits).mean(axis=1)
boot = errors[rng.integers(0,len(errors),size=(2000,len(errors)))].mean(axis=1)
print('Held-out person-average MAE:',errors.mean())
print('Person bootstrap 95% interval:',np.quantile(boot,[0.025,0.975]))
assert np.isfinite(boot).all()

Held-out person-average MAE: 0.3436994002647563
Person bootstrap 95% interval: [0.2937563  0.39793076]


## Explain without AI

Draw the full split diagram and explain why an inner grouped CV cannot detect every new-site shift. Report the failed shortcut alongside the successful simpler model. Decide whether the simpler feature set was specified before the test was seen; if not, label it exploratory. Propose a genuinely independent validation cohort and participant-level metrics.

<details><summary>Answer guide</summary>The shortcut carries site outcome differences in development but shifts to an unsupported value at the held-out site. A well-implemented pipeline can still generalize poorly. The stable feature is supplied as a prespecified comparison in this controlled simulation. A bootstrap over one site's people cannot estimate uncertainty across unseen sites.</details>

**Submission:** record the input, operation, parameters, output, one preserved property, one lost property, and the evidence that would make you reject the result. Include the prompt and any corrections you made to the generated code. A saved answer is not evidence of understanding until you can defend it orally.


### Return to the research question

Revisit [PD02](../../curriculum/papers/design.md#pd02), [PM03](../../curriculum/papers/modeling.md#pm03) and your initial prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure or section locator. Which part of the published result remains open after this exercise?
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Include this entry in the A2 portfolio when relevant.
